# F1 Pit Stop Prediction — Baseline EDA & Model
Kaggle Playground Series S6E5

## Imports

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

## Load Data

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head()

Train: (439140, 16)  |  Test: (188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


## Explore

In [4]:
train.info()
train.describe()

<class 'pandas.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  str    
 2   Compound                439140 non-null  str    
 3   Race                    439140 non-null  str    
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change         439140 

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


In [5]:
print("Nulls:")
print(train.isnull().sum())

print("\nTarget distribution:")
print(train["PitNextLap"].value_counts(normalize=True))

Nulls:
id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

Target distribution:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


## Prepare Features

In [16]:
X = train.drop(columns=["PitNextLap", "id", "Year"])
y = train["PitNextLap"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Valid: {X_valid.shape}")

Train: (351312, 15)  |  Valid: (87828, 15)


## Train CatBoost

In [17]:
categorical_features = ["Driver", "Compound", "Race"]

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid)
)

0:	test: 0.8680278	best: 0.8680278 (0)	total: 355ms	remaining: 2m 56s
100:	test: 0.9187678	best: 0.9187678 (100)	total: 10.1s	remaining: 39.7s
200:	test: 0.9280428	best: 0.9280428 (200)	total: 19.8s	remaining: 29.5s
300:	test: 0.9329148	best: 0.9329148 (300)	total: 28s	remaining: 18.5s
400:	test: 0.9355403	best: 0.9355403 (400)	total: 36.2s	remaining: 8.93s
499:	test: 0.9374551	best: 0.9374551 (499)	total: 44.7s	remaining: 0us

bestTest = 0.9374551065
bestIteration = 499



CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, verbose=100)

## Evaluate

In [8]:
preds = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, preds)
print(f"Validation AUC: {auc:.6f}")

Validation AUC: 0.937534


## Feature Importance

In [9]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

print(importance_df.to_string(index=False))

               Feature  Importance
         LapTime_Delta   21.446783
                 Stint   20.539426
              TyreLife   14.188058
          RaceProgress   12.771448
       Position_Change    7.008366
                  Race    5.582393
             LapNumber    5.165721
              Compound    4.066525
Cumulative_Degradation    3.053102
           LapTime (s)    2.405722
              Position    1.325444
                Driver    1.275592
               PitStop    1.171421


In [10]:
test_preds = model.predict_proba(test)[:, 1]
print(test_preds[:10])

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=3]="British Grand Prix": Cannot convert 'British Grand Prix' to float

In [14]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})
submission.to_csv("submission.csv", index=False)

In [12]:
train_features = train.drop(columns=["id", "Year", "PitNextLap"])
y = train["PitNextLap"]

X_train, X_valid, y_train, y_valid = train_test_split(
    train_features, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Valid: {X_valid.shape}")
categorical_features = ["Driver", "Compound", "Race"]

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid)
)

Train: (351312, 13)  |  Valid: (87828, 13)
0:	test: 0.8686916	best: 0.8686916 (0)	total: 197ms	remaining: 1m 38s
100:	test: 0.9198775	best: 0.9198775 (100)	total: 8.98s	remaining: 35.5s
200:	test: 0.9281115	best: 0.9281115 (200)	total: 17.3s	remaining: 25.7s
300:	test: 0.9329389	best: 0.9329389 (300)	total: 25.5s	remaining: 16.9s
400:	test: 0.9357282	best: 0.9357282 (400)	total: 33.9s	remaining: 8.37s
499:	test: 0.9375337	best: 0.9375337 (499)	total: 42.4s	remaining: 0us

bestTest = 0.9375337337
bestIteration = 499



CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, verbose=100)

## 🧠 FEATURE 2 — PaceDropRisk

In [13]:
# PaceDropRisk feature
# Captures pace deterioration + tyre degradation together

train["PaceDropRisk"] = (
    train["LapTime_Delta"] *
    train["Cumulative_Degradation"]
)

test["PaceDropRisk"] = (
    test["LapTime_Delta"] *
    test["Cumulative_Degradation"]
)

# Preview new feature
train[[
    "LapTime_Delta",
    "Cumulative_Degradation",
    "PaceDropRisk"
]].head()

,LapTime_Delta,Cumulative_Degradation,PaceDropRisk
0,-7.564,21.019,-158.987716
1,-32.617,-223.207,7280.342719
2,-7.540,-100.529,757.988660
3,-7.324,-7.324,53.640976
4,8.965,-14.139,-126.756135


## 🧠 FEATURE 3 — PositionRisk

In [14]:
# PositionRisk feature
# Combines track position with tyre age

train["PositionRisk"] = (
    train["Position"] *
    train["TyreLife"]
)

test["PositionRisk"] = (
    test["Position"] *
    test["TyreLife"]
)

# Preview new feature
train[["Position", "TyreLife", "PositionRisk"]].head()

,Position,TyreLife,PositionRisk
0,8,39.0,312.0
1,4,7.0,28.0
2,13,22.0,286.0
3,7,2.0,14.0
4,2,6.0,12.0


In [ ]:
X = train.drop(columns=["PitNextLap", "id", "Year"])

model.fit()
roc_auc_score()

ValueError: At least one array required as input

In [18]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier


# =========================================
# LOAD DATA
# =========================================

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")


# =========================================
# FEATURE ENGINEERING
# =========================================

# -----------------------------------------
# Feature 1: TyrePressure
# Combines tyre age with race progression
# -----------------------------------------

train["TyrePressure"] = (
    train["TyreLife"] * train["RaceProgress"]
)

test["TyrePressure"] = (
    test["TyreLife"] * test["RaceProgress"]
)


# -----------------------------------------
# Feature 2: PaceDropRisk
# Captures pace drop + tyre degradation
# -----------------------------------------

train["PaceDropRisk"] = (
    train["LapTime_Delta"] *
    train["Cumulative_Degradation"]
)

test["PaceDropRisk"] = (
    test["LapTime_Delta"] *
    test["Cumulative_Degradation"]
)


# -----------------------------------------
# Feature 3: PositionRisk
# Combines position with tyre age
# -----------------------------------------

train["PositionRisk"] = (
    train["Position"] *
    train["TyreLife"]
)

test["PositionRisk"] = (
    test["Position"] *
    test["TyreLife"]
)


# =========================================
# CREATE FEATURES (X) AND TARGET (y)
# =========================================

# Remove:
# - PitNextLap (target)
# - id (identifier)
# - Year (suspicious feature)

X = train.drop(columns=["PitNextLap", "id", "Year"])

y = train["PitNextLap"]


# Test features must match training features
test_features = test.drop(columns=["id", "Year"])


# =========================================
# TRAIN / VALIDATION SPLIT
# =========================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =========================================
# DEFINE CATEGORICAL FEATURES
# =========================================

categorical_features = [
    "Driver",
    "Compound",
    "Race"
]


# =========================================
# BUILD CATBOOST MODEL
# =========================================

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    verbose=100
)


# =========================================
# TRAIN MODEL
# =========================================

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid)
)


# =========================================
# VALIDATION PREDICTIONS
# =========================================

valid_preds = model.predict_proba(X_valid)[:, 1]


# =========================================
# EVALUATE MODEL
# =========================================

auc = roc_auc_score(y_valid, valid_preds)

print("Validation AUC:", auc)


# =========================================
# FEATURE IMPORTANCE
# =========================================

feature_importance = model.get_feature_importance()

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": feature_importance
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df)


# =========================================
# GENERATE TEST PREDICTIONS
# =========================================

test_preds = model.predict_proba(test_features)[:, 1]


# =========================================
# CREATE SUBMISSION FILE
# =========================================

submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})


# =========================================
# SAVE SUBMISSION
# =========================================

submission.to_csv("submission_day4.csv", index=False)

print("Submission file saved successfully!")

0:	test: 0.8669867	best: 0.8669867 (0)	total: 177ms	remaining: 1m 28s
100:	test: 0.9191668	best: 0.9191668 (100)	total: 9.42s	remaining: 37.2s
200:	test: 0.9277937	best: 0.9277937 (200)	total: 18.6s	remaining: 27.6s
300:	test: 0.9325808	best: 0.9325808 (300)	total: 27.3s	remaining: 18.1s
400:	test: 0.9354440	best: 0.9354440 (400)	total: 37s	remaining: 9.13s
499:	test: 0.9372660	best: 0.9372660 (499)	total: 47.1s	remaining: 0us

bestTest = 0.9372660111
bestIteration = 499

Validation AUC: 0.9372660111040433
                   Feature  Importance
5                    Stint   21.957050
9            LapTime_Delta   19.719951
6                 TyreLife   13.308723
11            RaceProgress   11.704901
12         Position_Change    5.862697
2                     Race    5.671075
4                LapNumber    4.730417
1                 Compound    4.690624
14            PaceDropRisk    3.655740
10  Cumulative_Degradation    2.615555
8              LapTime (s)    2.136848
0                   

In [19]:
# =========================================
# TUNED CATBOOST MODEL
# =========================================

tuned_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    eval_metric="AUC",
    loss_function="Logloss",
    random_seed=42,
    verbose=100
)


# =========================================
# TRAIN MODEL
# =========================================

tuned_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid)
)


# =========================================
# VALIDATION PREDICTIONS
# =========================================

tuned_valid_preds = tuned_model.predict_proba(X_valid)[:, 1]


# =========================================
# EVALUATE MODEL
# =========================================

tuned_auc = roc_auc_score(y_valid, tuned_valid_preds)

print("Tuned Validation AUC:", tuned_auc)


# =========================================
# FEATURE IMPORTANCE
# =========================================

tuned_feature_importance = tuned_model.get_feature_importance()

tuned_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": tuned_feature_importance
})

tuned_importance_df = tuned_importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(tuned_importance_df)


# =========================================
# GENERATE TEST PREDICTIONS
# =========================================

tuned_test_preds = tuned_model.predict_proba(test_features)[:, 1]


# =========================================
# CREATE FINAL SUBMISSION
# =========================================

final_submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": tuned_test_preds
})


# =========================================
# SAVE FINAL SUBMISSION
# =========================================

final_submission.to_csv(
    "submission_final_tuned.csv",
    index=False
)

print("Final tuned submission saved!")

0:	test: 0.8817044	best: 0.8817044 (0)	total: 458ms	remaining: 7m 37s
100:	test: 0.9189438	best: 0.9189438 (100)	total: 11.9s	remaining: 1m 46s
200:	test: 0.9277115	best: 0.9277115 (200)	total: 22.3s	remaining: 1m 28s
300:	test: 0.9324402	best: 0.9324402 (300)	total: 33s	remaining: 1m 16s
400:	test: 0.9352084	best: 0.9352084 (400)	total: 43.9s	remaining: 1m 5s
500:	test: 0.9375118	best: 0.9375118 (500)	total: 56.6s	remaining: 56.4s
600:	test: 0.9391384	best: 0.9391384 (600)	total: 1m 8s	remaining: 45.5s
700:	test: 0.9402689	best: 0.9402689 (700)	total: 1m 22s	remaining: 35.2s
800:	test: 0.9411566	best: 0.9411566 (800)	total: 1m 39s	remaining: 24.6s
900:	test: 0.9418735	best: 0.9418735 (900)	total: 1m 51s	remaining: 12.3s
999:	test: 0.9424257	best: 0.9424257 (999)	total: 2m 4s	remaining: 0us

bestTest = 0.9424256886
bestIteration = 999

Tuned Validation AUC: 0.9424256885608271
                   Feature  Importance
9            LapTime_Delta   17.855364
5                    Stint   17.7